# Unified benchmark: MobileNetV2 vs EfficientNetB0 vs Hybrid EfficientNetB0–Transformer
**LC25000 lung subset (lung_aca, lung_n, lung_scc)**: one shared split, one shared training protocol.

**How to run (Google Colab):**
1. *Runtime → Change runtime type → T4 GPU*.
2. *Runtime → Run all*. Total time is roughly 45–75 minutes.
3. The last cell prints a **summary** and creates **`unified_results.zip`**.
   Send both to Claude.

Everything that differed between the three original notebooks is now identical:
split, image size, batch size, epochs, learning rates, patience, augmentation and precision (float32 for all).

### Settings (identical for all three models)

In [ ]:
import os
TEST_MODE = os.environ.get("UNIFIED_TEST_MODE") == "1"   # used only for a quick offline test; leave as is

SEED            = 42
IMG_SIZE        = 160
BATCH_SIZE      = 16
EPOCHS_STAGE1   = 8          # frozen backbone
EPOCHS_STAGE2   = 8          # fine-tuning
LR_STAGE1       = 1e-4
LR_STAGE2       = 5e-5
UNFREEZE_LAST_N = 40         # backbone layers unfrozen in Stage 2 (BatchNorm layers stay frozen)
PATIENCE        = 5
DROPOUT         = 0.3
XAI_PER_CLASS   = 5          # correctly classified test images per class shown in XAI figures
XAI_STATS_N     = 50         # test images per class used for XAI statistics
LAT_WARMUP, LAT_RUNS = 20, 100
WEIGHTS         = "imagenet"
OUT_DIR         = "/content/unified_results"

# Where the LC25000 data is. Leave DATA_DIR = None to download it automatically from Kaggle.
# If you already have it on Google Drive, set e.g.
# DATA_DIR = "/content/drive/MyDrive/lung_colon_image_set/lung_image_sets"
DATA_DIR = None

if TEST_MODE:
    EPOCHS_STAGE1, EPOCHS_STAGE2, XAI_PER_CLASS, XAI_STATS_N = 1, 1, 2, 4
    LAT_WARMUP, LAT_RUNS, WEIGHTS = 2, 5, None
    DATA_DIR = os.environ["UNIFIED_TEST_DATA"]
    OUT_DIR = os.environ["UNIFIED_TEST_OUT"]

### Imports and reproducibility

In [ ]:
import json, time, hashlib, random, shutil, glob
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg") if TEST_MODE else None
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc,
                             precision_recall_curve, average_precision_score, f1_score)
from statsmodels.stats.contingency_tables import mcnemar

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
keras.mixed_precision.set_global_policy("float32")
os.makedirs(OUT_DIR, exist_ok=True)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

### Get the data

In [ ]:
CLASSES = ["lung_aca", "lung_n", "lung_scc"]

def find_lung_dir(root):
    for dirpath, dirnames, _ in os.walk(root):
        if all(c in dirnames for c in CLASSES):
            return dirpath
    raise FileNotFoundError(f"No folder containing {CLASSES} found under {root}")

if DATA_DIR is None:
    import kagglehub
    root = kagglehub.dataset_download("andrewmvd/lung-and-colon-cancer-histopathological-images")
    DATA_DIR = find_lung_dir(root)
elif not all(os.path.isdir(os.path.join(DATA_DIR, c)) for c in CLASSES):
    if DATA_DIR.startswith("/content/drive"):
        from google.colab import drive; drive.mount("/content/drive")
    DATA_DIR = find_lung_dir(DATA_DIR)
print("Data folder:", DATA_DIR)

### ONE shared MD5-deduplicated, stratified 70/15/15 split (built once, used by all three models)

In [ ]:
rows = []
for label, c in enumerate(CLASSES):
    for f in sorted(glob.glob(os.path.join(DATA_DIR, c, "*"))):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            with open(f, "rb") as fh:
                rows.append({"path": f, "label": label, "cls": c, "md5": hashlib.md5(fh.read()).hexdigest()})
df_all = pd.DataFrame(rows)
df = df_all.drop_duplicates("md5", keep="first").reset_index(drop=True)
# an image whose identical copy sits in another class would be ambiguous; drop such hashes entirely
conflict = df_all.groupby("md5")["label"].nunique()
df = df[~df["md5"].isin(conflict[conflict > 1].index)].reset_index(drop=True)
print(f"Images found: {len(df_all)} | exact duplicates removed: {len(df_all) - len(df)} | unique kept: {len(df)}")

train_df, tmp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df["label"], random_state=SEED)
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part.assign(split=name).to_csv(os.path.join(OUT_DIR, f"split_{name}.csv"), index=False)
overlap = (set(train_df.md5) & set(val_df.md5)) | (set(train_df.md5) & set(test_df.md5)) | (set(val_df.md5) & set(test_df.md5))
assert not overlap, "hash overlap between splits!"
split_info = {s: {"n": len(p), **p["cls"].value_counts().reindex(CLASSES).to_dict()}
              for s, p in [("train", train_df), ("val", val_df), ("test", test_df)]}
split_info["duplicates_removed"] = int(len(df_all) - len(df))
print(json.dumps(split_info, indent=1))

### Shared tf.data pipeline (decode+resize is cached BEFORE augmentation, so augmentation is fresh every epoch)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(15 / 360),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.1),
], name="augment")

def load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.round(img), tf.uint8), tf.one_hot(label, len(CLASSES))

def make_ds(part, training):
    ds = tf.data.Dataset.from_tensor_slices((part["path"].values, part["label"].values))
    ds = ds.map(load, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(part), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df, False), make_ds(test_df, False)
y_test = test_df["label"].values

### Model builders (functional models, so Grad-CAM layers are directly reachable)

In [ ]:
def head(x, name):
    x = layers.Dropout(DROPOUT, name="dropout")(x)
    logits = layers.Dense(len(CLASSES), name="logits")(x)
    out = layers.Activation("softmax", name="pred")(logits)
    return out

def build_mobilenet():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = layers.Rescaling(1 / 127.5, offset=-1, name="mnv2_preprocess")(inp)   # = mobilenet_v2.preprocess_input
    base = keras.applications.MobileNetV2(include_top=False, weights=WEIGHTS, input_tensor=x)
    x = layers.GlobalAveragePooling2D(name="gap")(base.output)
    return keras.Model(inp, head(x, "MobileNetV2"), name="MobileNetV2"), base, "out_relu"

def build_efficientnet():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")   # EfficientNetB0 rescales internally
    base = keras.applications.EfficientNetB0(include_top=False, weights=WEIGHTS, input_tensor=inp)
    x = layers.GlobalAveragePooling2D(name="gap")(base.output)
    return keras.Model(inp, head(x, "EfficientNetB0"), name="EfficientNetB0"), base, "top_activation"

def build_hybrid():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    base = keras.applications.EfficientNetB0(include_top=False, weights=WEIGHTS, input_tensor=inp)
    fmap = base.output                                   # 5x5x1280
    h, w, c = fmap.shape[1], fmap.shape[2], fmap.shape[3]
    tokens = layers.Reshape((h * w, c), name="tokens")(fmap)
    t = layers.LayerNormalization(epsilon=1e-6, name="ln1")(tokens)
    attn_out, attn_scores = layers.MultiHeadAttention(num_heads=2, key_dim=16, name="mhsa")(
        t, t, return_attention_scores=True)
    x = layers.Add(name="res1")([tokens, attn_out])
    t = layers.LayerNormalization(epsilon=1e-6, name="ln2")(x)
    t = layers.Dense(128, activation="gelu", name="mlp1")(t)
    t = layers.Dense(c, name="mlp2")(t)
    x = layers.Add(name="res2")([x, t])
    x = layers.GlobalAveragePooling1D(name="gap")(x)
    model = keras.Model(inp, head(x, "Hybrid"), name="Hybrid_EffNetB0_Transformer")
    attn_model = keras.Model(inp, attn_scores, name="hybrid_attention")   # shares weights with model
    return model, base, "top_activation", attn_model

BUILDERS = {"MobileNetV2": build_mobilenet, "EfficientNetB0": build_efficientnet,
            "Hybrid_EffNetB0_Transformer": build_hybrid}

### Training (identical two-stage protocol for every model)

In [ ]:
def set_stage2(base):
    base.trainable = True
    for layer in base.layers[:-UNFREEZE_LAST_N]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

def callbacks(ckpt):
    return [keras.callbacks.ModelCheckpoint(ckpt, monitor="val_accuracy", save_best_only=True, save_weights_only=True),
            keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=PATIENCE, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)]

def count_params(ws):
    return int(sum(np.prod(w.shape) for w in ws))

trained, histories, params = {}, {}, {}
for name, builder in BUILDERS.items():
    print("\n" + "=" * 70 + f"\nTRAINING {name}\n" + "=" * 70)
    keras.backend.clear_session(); tf.random.set_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    built = builder()
    model, base, cam_layer = built[0], built[1], built[2]
    attn_model = built[3] if len(built) > 3 else None
    mdir = os.path.join(OUT_DIR, name); os.makedirs(mdir, exist_ok=True)
    ckpt = os.path.join(mdir, "best.weights.h5")

    base.trainable = False
    model.compile(keras.optimizers.Adam(LR_STAGE1), "categorical_crossentropy", metrics=["accuracy"])
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_STAGE1, callbacks=callbacks(ckpt), verbose=2)
    trainable_s1 = count_params(model.trainable_weights)

    set_stage2(base)
    model.compile(keras.optimizers.Adam(LR_STAGE2), "categorical_crossentropy", metrics=["accuracy"])
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_STAGE2, callbacks=callbacks(ckpt), verbose=2)
    model.load_weights(ckpt)   # best validation-accuracy weights across both stages

    histories[name] = {k: h1.history[k] + h2.history[k] for k in h1.history if k in h2.history}
    histories[name]["stage1_epochs"] = len(h1.history["loss"])
    params[name] = {"total": model.count_params(), "trainable_stage1": trainable_s1,
                    "trainable_stage2": count_params(model.trainable_weights)}
    trained[name] = {"model": model, "cam_layer": cam_layer, "attn_model": attn_model, "dir": mdir}
    json.dump(histories[name], open(os.path.join(mdir, "history.json"), "w"), indent=1)
    print(name, params[name])

### Evaluation on the SAME test set for every model

In [ ]:
def ece_score(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    bins, e = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)

def bootstrap_ci(correct, n=2000):
    rng = np.random.default_rng(SEED)
    accs = [correct[rng.integers(0, len(correct), len(correct))].mean() for _ in range(n)]
    return [float(np.percentile(accs, 2.5)), float(np.percentile(accs, 97.5))]

summary, preds = {}, {}
for name, t in trained.items():
    model = t["model"]
    probs = model.predict(test_ds, verbose=0)
    yp = probs.argmax(1)
    preds[name] = yp
    loss = float(keras.losses.categorical_crossentropy(np.eye(len(CLASSES))[y_test], probs).numpy().mean())
    rep = classification_report(y_test, yp, target_names=CLASSES, digits=4, output_dict=True)
    cm = confusion_matrix(y_test, yp)
    pd.DataFrame({"path": test_df["path"].values, "y_true": y_test, "y_pred": yp,
                  **{f"p_{c}": probs[:, i] for i, c in enumerate(CLASSES)}}).to_csv(
        os.path.join(t["dir"], "test_predictions.csv"), index=False)
    t["probs"] = probs
    summary[name] = {
        "test_n": int(len(y_test)), "accuracy": float((yp == y_test).mean()),
        "accuracy_95ci": bootstrap_ci((yp == y_test).astype(float)),
        "macro_f1": float(f1_score(y_test, yp, average="macro")), "loss": loss,
        "misclassified": int((yp != y_test).sum()), "ece": ece_score(probs, y_test),
        "per_class": {c: {k: round(rep[c][k], 4) for k in ("precision", "recall", "f1-score")} for c in CLASSES},
        "auc": {c: float(auc(*roc_curve(y_test == i, probs[:, i])[:2])) for i, c in enumerate(CLASSES)},
        "confusion_matrix": cm.tolist(), "params": params[name],
        "epochs_run": len(histories[name]["loss"]), "stage1_epochs": histories[name]["stage1_epochs"],
    }
    print(f"{name}: acc={summary[name]['accuracy']:.4f}  errors={summary[name]['misclassified']}  ECE={summary[name]['ece']:.4f}")

# Pairwise exact McNemar tests (possible now because the test set is shared)
names = list(trained)
mcn = {}
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = preds[names[i]] == y_test, preds[names[j]] == y_test
        table = [[int((a & b).sum()), int((a & ~b).sum())], [int((~a & b).sum()), int((~a & ~b).sum())]]
        res = mcnemar(table, exact=True)
        mcn[f"{names[i]} vs {names[j]}"] = {"table": table, "p_value": float(res.pvalue)}
print(json.dumps(mcn, indent=1))

### Figures: curves, confusion matrices, ROC, PR, per-class bars, reliability diagrams (one row of 3 panels each)

In [ ]:
def panel(fn, fname, title):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
    for ax, name in zip(axes, names):
        fn(ax, name); ax.set_title(name.replace("_", " "))
    fig.suptitle(title); fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, fname), dpi=300, bbox_inches="tight"); plt.close(fig)

def curve(metric):
    def f(ax, name):
        h = histories[name]; ep = np.arange(1, len(h[metric]) + 1)
        ax.plot(ep, h[metric], label="train"); ax.plot(ep, h["val_" + metric], label="validation")
        ax.axvline(h["stage1_epochs"] + 0.5, ls="--", c="gray", lw=1, label="start of fine-tuning")
        ax.set_xlabel("epoch"); ax.set_ylabel(metric); ax.legend()
    return f

def cm_plot(ax, name):
    cm = np.array(summary[name]["confusion_matrix"])
    ax.imshow(cm, cmap="Blues")
    for (r, c), v in np.ndenumerate(cm):
        ax.text(c, r, v, ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
    ax.set_xticks(range(3), CLASSES); ax.set_yticks(range(3), CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")

def roc_plot(ax, name):
    p = trained[name]["probs"]
    for i, c in enumerate(CLASSES):
        fpr, tpr, _ = roc_curve(y_test == i, p[:, i]); ax.plot(fpr, tpr, label=f"{c} (AUC={auc(fpr, tpr):.4f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.legend()

def pr_plot(ax, name):
    p = trained[name]["probs"]
    for i, c in enumerate(CLASSES):
        pr, rc, _ = precision_recall_curve(y_test == i, p[:, i])
        ax.plot(rc, pr, label=f"{c} (AP={average_precision_score(y_test == i, p[:, i]):.4f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.legend()

def bar_plot(ax, name):
    pc = summary[name]["per_class"]; x = np.arange(3)
    for k, (m, lab) in enumerate([("precision", "Precision"), ("recall", "Recall"), ("f1-score", "F1")]):
        ax.bar(x + (k - 1) * 0.25, [pc[c][m] for c in CLASSES], 0.25, label=lab)
    ax.set_xticks(x, CLASSES); ax.set_ylim(0.90, 1.01); ax.legend(loc="lower right")

def rel_plot(ax, name):
    p = trained[name]["probs"]; conf, pred = p.max(1), p.argmax(1)
    bins = np.linspace(0, 1, 16); mids, accs = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): mids.append(conf[m].mean()); accs.append((pred[m] == y_test[m]).mean())
    ax.plot([0, 1], [0, 1], "k--", lw=1); ax.plot(mids, accs, "o-")
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy"); ax.text(0.05, 0.9, f"ECE = {summary[name]['ece']:.4f}")

panel(curve("accuracy"), "fig_accuracy_curves.png", "Training vs validation accuracy")
panel(curve("loss"), "fig_loss_curves.png", "Training vs validation loss")
panel(cm_plot, "fig_confusion_matrices.png", f"Confusion matrices (shared test set, n = {len(y_test)})")
panel(roc_plot, "fig_roc.png", "Per-class ROC curves")
panel(pr_plot, "fig_pr.png", "Per-class precision–recall curves")
panel(bar_plot, "fig_per_class_metrics.png", "Per-class precision, recall and F1")
panel(rel_plot, "fig_reliability.png", "Reliability diagrams")

### Latency (compiled forward pass, batch size 1) and FLOPs

In [ ]:
lat = {}
x1 = tf.constant(np.random.default_rng(SEED).uniform(0, 255, (1, IMG_SIZE, IMG_SIZE, 3)).astype("float32"))
for name, t in trained.items():
    m = t["model"]
    fwd = tf.function(lambda x, m=m: m(x, training=False))
    for _ in range(LAT_WARMUP): fwd(x1).numpy()
    ts = []
    for _ in range(LAT_RUNS):
        s = time.perf_counter(); fwd(x1).numpy(); ts.append((time.perf_counter() - s) * 1000)
    lat[name] = {"median_ms": float(np.median(ts)), "mean_ms": float(np.mean(ts)), "sd_ms": float(np.std(ts)),
                 "p95_ms": float(np.percentile(ts, 95))}
    try:
        from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2_as_graph
        cf = tf.function(lambda x, m=m: m(x, training=False)).get_concrete_function(
            tf.TensorSpec([1, IMG_SIZE, IMG_SIZE, 3], tf.float32))
        _, graph_def = convert_variables_to_constants_v2_as_graph(cf)
        with tf.Graph().as_default() as g:
            tf.graph_util.import_graph_def(graph_def, name="")
            opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
            opts["output"] = "none"
            fl = tf.compat.v1.profiler.profile(graph=g, run_meta=tf.compat.v1.RunMetadata(), cmd="op", options=opts)
        lat[name]["gflops"] = round(fl.total_float_ops / 1e9, 4)   # FLOPs (multiply and add counted separately)
    except Exception as e:
        lat[name]["gflops"] = None; lat[name]["flops_error"] = str(e)[:200]
    summary[name]["efficiency"] = lat[name]
    print(name, lat[name])

> **Note (superseded section).** The explainability cell below was used only for an early exploratory analysis. Its Grad-CAM++ implementation raises the exponential term to the second and third powers, which differs from the closed form of Chattopadhay et al. (2018). **None of the explainability results reported in the paper come from this cell.** All reported saliency results (Table 2 and Figure 3) were produced with `XAI_Corrected_Analysis.ipynb`. The cell is retained unchanged so that this notebook reproduces the original run exactly.

### Explainability: Grad-CAM and Grad-CAM++ (all models), attention map (hybrid)

In [ ]:
def cam_maps(model, layer_name, img, class_idx):
    gm = keras.Model(model.inputs, [model.get_layer(layer_name).output, model.get_layer("logits").output])
    with tf.GradientTape() as tape:
        A, logits = gm(img[None], training=False)
        score = logits[:, class_idx]
    g = tape.gradient(score, A)[0].numpy(); A = A[0].numpy()
    gc = np.maximum((A * g.mean((0, 1))).sum(-1), 0)                          # Grad-CAM
    raw_min_max = (float((A * g.mean((0, 1))).sum(-1).min()), float((A * g.mean((0, 1))).sum(-1).max()))
    eg = np.exp(float(score[0]))                                              # Grad-CAM++ (exp-score form)
    g1, g2, g3 = eg * g, (eg * g) ** 2, (eg * g) ** 3
    denom = 2 * g2 + A.sum((0, 1)) * g3
    alpha = g2 / np.where(denom != 0, denom, 1)
    w = (alpha * np.maximum(g1, 0)).sum((0, 1))
    gpp = np.maximum((A * w).sum(-1), 0)
    return gc, gpp, raw_min_max

def attn_map(attn_model, img):
    s = attn_model(img[None], training=False).numpy()[0]       # heads x query x key
    rec = s.mean(0).mean(0)                                    # attention received per token
    n = int(np.sqrt(rec.size)); return rec.reshape(n, n)

def norm(m):
    m = m - m.min(); return m / m.max() if m.max() > 0 else m

def overlay(ax, img, m, title):
    ax.imshow(img.astype("uint8"))
    ax.imshow(tf.image.resize(norm(m)[..., None], (IMG_SIZE, IMG_SIZE)).numpy()[..., 0], cmap="jet", alpha=0.4)
    ax.set_title(title, fontsize=8); ax.axis("off")

test_imgs = np.concatenate([x.numpy() for x, _ in test_ds])
xai = {}
for name, t in trained.items():
    model, yp = t["model"], preds[name]
    correct_idx = {i: np.where((y_test == i) & (yp == i))[0] for i in range(3)}
    # statistics over XAI_STATS_N correctly classified images per class
    st = {"gradcam_blank": 0, "n": 0, "gradcampp_peaks": [], "attn_peaks": []}
    for i in range(3):
        for k in correct_idx[i][:XAI_STATS_N]:
            gc, gpp, _ = cam_maps(model, t["cam_layer"], test_imgs[k], i)
            st["n"] += 1; st["gradcam_blank"] += int(gc.max() == 0)
            st["gradcampp_peaks"].append(int(np.argmax(gpp)))
            if t["attn_model"] is not None:
                st["attn_peaks"].append(int(np.argmax(attn_map(t["attn_model"], test_imgs[k]))))
    xai[name] = {"images_analysed": st["n"], "gradcam_blank_maps": st["gradcam_blank"],
                 "gradcampp_distinct_peak_positions": len(set(st["gradcampp_peaks"])),
                 "gradcampp_most_common_peak_share": (max(np.bincount(st["gradcampp_peaks"])) / st["n"]) if st["n"] else None}
    if st["attn_peaks"]:
        xai[name]["attention_distinct_peak_positions"] = len(set(st["attn_peaks"]))
        xai[name]["attention_most_common_peak_share"] = max(np.bincount(st["attn_peaks"])) / st["n"]
    # figure: rows = images, columns = original / Grad-CAM / Grad-CAM++ / (attention)
    cols = 4 if t["attn_model"] is not None else 3
    sel = [(i, k) for i in range(3) for k in correct_idx[i][:XAI_PER_CLASS]]
    fig, axes = plt.subplots(len(sel), cols, figsize=(3 * cols, 3 * len(sel)), squeeze=False)
    for r, (i, k) in enumerate(sel):
        img = test_imgs[k]; gc, gpp, _ = cam_maps(model, t["cam_layer"], img, i)
        axes[r, 0].imshow(img.astype("uint8")); axes[r, 0].set_title(CLASSES[i], fontsize=8); axes[r, 0].axis("off")
        overlay(axes[r, 1], img, gc, "Grad-CAM" + (" (blank)" if gc.max() == 0 else ""))
        overlay(axes[r, 2], img, gpp, "Grad-CAM++")
        if cols == 4: overlay(axes[r, 3], img, attn_map(t["attn_model"], img), "Self-attention")
    fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, f"fig_xai_{name}.png"), dpi=200, bbox_inches="tight"); plt.close(fig)
    summary[name]["xai"] = xai[name]
    print(name, xai[name])

### Save everything and print the summary to send to Claude

In [ ]:
settings = {k: globals()[k] for k in ["SEED", "IMG_SIZE", "BATCH_SIZE", "EPOCHS_STAGE1", "EPOCHS_STAGE2", "LR_STAGE1",
                                      "LR_STAGE2", "UNFREEZE_LAST_N", "PATIENCE", "DROPOUT"]}
full = {"settings": settings, "split": split_info, "models": summary, "mcnemar": mcn,
        "tf_version": tf.__version__, "gpu": [d.name for d in tf.config.list_physical_devices("GPU")]}
json.dump(full, open(os.path.join(OUT_DIR, "summary.json"), "w"), indent=1, default=float)
shutil.make_archive(OUT_DIR.rstrip("/"), "zip", OUT_DIR)

print("\n" + "=" * 70 + "\nSUMMARY — copy everything below and send it to Claude\n" + "=" * 70)
print(json.dumps(full, indent=1, default=float))
if not TEST_MODE:
    try:
        from google.colab import files; files.download(OUT_DIR.rstrip("/") + ".zip")
    except Exception:
        print("Download unified_results.zip from the Files panel on the left.")

### Show the main figures here in Colab

In [ ]:
if not TEST_MODE:
    from IPython.display import Image as _I, display
    for f in ["fig_confusion_matrices.png", "fig_accuracy_curves.png", "fig_xai_Hybrid_EffNetB0_Transformer.png"]:
        display(_I(os.path.join(OUT_DIR, f), width=1000))